# Exercise 2. Annotate and Classify on Few Examples
So far we have been working with pre-labeled datasets. However, many interesting NLP questions are also to be found in datasets that do not have any labels!

**Let's imagine that we want to expand our student questions dataset** with *history* and *social science* questions:

> Who led the civil rights movement in the United States? 
> A. Nelson Mandela
> B. Martin Luther King Jr.
> C. Malcolm X
> D. Rosa Parks

## 2.1 Let's Annotate!
:::{admonition} HANDS-ON
:class: red
Below this box, there will be 16 questions that you need to annotate as either "Social Science" or "History". Read through them carefully and annotate by yourself. Then compare with a friend. Consider these: 
- Do you agree on the annotations? 
- Were there any difficult ones?

I have tried to design the questions so that there is 8 per class, but you might disagree here!
:::


Copy the dictionary of questions into your own notebook/script and replace "LABEL" with `"Social Science"` or `"History"`:

In [66]:
questions = {
    "Societies collapse when their political institutions fail. Do you agree?": "LABEL",
    "Can a country have elections but not be democratic? Argue for your answer.": "LABEL",
    "Who led the civil rights movement in the United States? \nA. Nelson Mandela\nB. Martin Luther King Jr.\nC. Malcolm X\nD. Rosa Parks": "LABEL",
    "When empires expand, is it always for economic reasons? Discuss with examples.": "LABEL",
    "Which organization was formed after World War II to promote peace and cooperation?\nA. League of Nations\nB. NATO\nC. United Nations\nD. European Union": "LABEL",
    "What is the main difference between capitalism and socialism?": "LABEL",
    "Nationalism can emerge from social institutions or historical events. Which factor do you think has been more influential in shaping modern states?": "LABEL",
    "Social progress does not always follow technological progress. Provide one example where technology increased inequality rather than reducing it.": "LABEL",
    "Which factor most contributed to rapid urban growth in 19th-century Europe?\nA. Agricultural decline\nB. Industrial job opportunities\nC. Rise of universities\nD. Religious reforms": "LABEL",
    "Social movements can succeed through protests, legal change, or public opinion. Which has been most effective? Explain with one example.": "LABEL",
    "How did the introduction of cash crops in colonial Africa reshape local social structures?\nA. Increased wealth for local farmers\nB. Strengthened traditional hierarchies\nC. Created labor inequalities\nD. All of the above": "LABEL",
    "Which factor contributed most to the spread of Islam across Africa and Asia before 1500 CE?\nA. Trade networks\nB. Military conquest\nC. Cultural assimilation\nD. Missionary activity": "LABEL",
    "Which technological innovation in the Indian Ocean trade had the largest societal effect before 1600 CE?\nA. Lateen sails\nB. Compass navigation\nC. Shipbuilding techniques\nD. Port infrastructure": "LABEL",
    "Which has a stronger influence on social behavior: historical narratives or contemporary media? Explain briefly.": "LABEL",
    "Which is more effective in promoting public health: top-down government interventions or grassroots social initiatives? Explain.": "LABEL",
    "Which factor most influences political participation in modern societies?\nA. Economic stability\nB. Education\nC. Media exposure\nD. Social networks": "LABEL"
}

In [67]:
# MY ANNOTATIONS
questions = {
    "Societies collapse when their political institutions fail. Do you agree?": "Social Science",
    "Can a country have elections but not be democratic? Argue for your answer.": "Social Science",
    "Who led the civil rights movement in the United States? \nA. Nelson Mandela\nB. Martin Luther King Jr.\nC. Malcolm X\nD. Rosa Parks": "History",
    "When empires expand, is it always for economic reasons? Discuss with examples.": "History",
    "Which organization was formed after World War II to promote peace and cooperation?\nA. League of Nations\nB. NATO\nC. United Nations\nD. European Union": "History",
    "What is the main difference between capitalism and socialism?": "Social Science",
    "Nationalism can emerge from social institutions or historical events. Which factor do you think has been more influential in shaping modern states?": "History",
    "Social progress does not always follow technological progress. Provide one example where technology increased inequality rather than reducing it.": "Social Science",
    "Which factor most contributed to rapid urban growth in 19th-century Europe?\nA. Agricultural decline\nB. Industrial job opportunities\nC. Rise of universities\nD. Religious reforms": "History",
    "Social movements can succeed through protests, legal change, or public opinion. Which has been most effective? Explain with one example.": "Social Science",
    "How did the introduction of cash crops in colonial Africa reshape local social structures?\nA. Increased wealth for local farmers\nB. Strengthened traditional hierarchies\nC. Created labor inequalities\nD. All of the above": "History",
    "Which factor contributed most to the spread of Islam across Africa and Asia before 1500 CE?\nA. Trade networks\nB. Military conquest\nC. Cultural assimilation\nD. Missionary activity": "History",
    "Which technological innovation in the Indian Ocean trade had the largest societal effect before 1600 CE?\nA. Lateen sails\nB. Compass navigation\nC. Shipbuilding techniques\nD. Port infrastructure": "History",
    "Which has a stronger influence on social behavior: historical narratives or contemporary media? Explain briefly.": "Social Science",
    "Which is more effective in promoting public health: top-down government interventions or grassroots social initiatives? Explain.": "Social Science",
    "Which factor most influences political participation in modern societies?\nA. Economic stability\nB. Education\nC. Media exposure\nD. Social networks": "Social Science"
}

:::{admonition} QUESTION
:class: red
The questions above were generated by ChatGPT. List any concerns that there may be with this approach.
:::

```{admonition} LLM FRAMING: How did I create the questions with ChatGPT?
:class: dropdown, fuchsia
I created these questions with ChatGPT (in the interface) by giving it: 
- Instructions about the original dataset (name of the dataset, on Kaggle, mix of multiple choice Q's, open-ended questions etc.)
- Instructions for what I wanted (history + social science questions, ideally some edge-cases)
- Four examples of questions from the original dataset

I also wrote in the same thread for a long time ... 
- I got ChatGPT to generate a lot of examples, then picked the ones that I liked the most - for the purpose of this class!
- I provided extra instructions such as "give me a few longer questions"

Synthetic data creation is not typically as trivial as my experimenting! This is a huge field on the rise with many considerations on how to do it best.
```

## 2.2 Setup: Install Additional Packages
Please download the packages below (in `venv` or in UCloud) in your terminal:
```bash
pip install sentence-transformers datasets transformers setfit
```

:::{admonition} Download in Jupyter
:class: tip
Remember, you can also download the packages in Jupyter with the `%pip` magic command as we have done in previous classes.
:::

I'll re-import everything (since you worked in a script last), but as always, import only what you need

In [152]:
from pathlib import Path
from datasets import load_dataset, ClassLabel, Dataset, concatenate_datasets, Value # note I have added Dataset, ConcatenateDataset and Value
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from setfit import SetFitModel, Trainer, TrainingArguments

## 2.3 Prepare Data

### New Data

We should now have a dictionary with a `question` as the `key` and its text label `Social Science` or `History` as the value. We also want numerical variables.

:::{admonition} HANDS-ON
:class: red
To recap for loops, create a numerical label mapping! You can:
- Define an empty list outside of the loop called `numeric_labels`
- Define a for loop that goes through each value (text label) of the dictionary
    - Checks if the value is `History` and appends the number `5` to `numeric_labels`
    - Checks if the value is `Social Science` and appends the number `6`
    - Has a final `else` statement that appends `-1` AND prints "Something went wrong, appending label -1"

*Optionally, you could consider if there are other ways than for loops for this!*
:::

Solution:

In [92]:
# create a list to store numeric labels
numeric_labels = []

# loop through the values and use if statements to assign numbers
for label in questions.values():
    if label == "History":
        numeric_labels.append(5)
    elif label == "Social Science":
        numeric_labels.append(6)
    else:
        print(f"[WARNING] Unknown label: {label}. Assigning -1.")
        numeric_labels.append(-1)  # or any number for unknown labels

Now that you have the variable `numeric_labels`, let's create a new dataset with our data!

In [ ]:
# format suitable for Dataset
new_train_data = {
    "text": list(questions.keys()),
    "label": list(numeric_labels) ,  
    "label_text": list(questions.values())   
}

new_train_ds = Dataset.from_dict(new_train_data)

For the validation dataset, I have made ChatGPT create and annotate a dataset that we'll load;

In [122]:
path = Path.cwd()
data_path = path.parents[1] / "resources" / "data" / "history_socsci_qs_val.csv"
new_val_ds = load_dataset("csv", data_files=str(data_path))["train"]

```{warning}
In an exam scenario, this type of dataset should be validated before use and you should be able to show and reflect upon any potential issues!
```

### Combine with Previous Data

We have annotated data, but let's prepare it with our previous ds! We'll load it again (feel free to not if you have it in a notebook):

In [111]:
data_path = path.parents[1] / "resources" / "data" / "hf" # path for huggingface datasets (so we don't have to redownload them every time)
ds = load_dataset("SetFit/student-question-categories", split="train", cache_dir=data_path)

Repo card metadata block was not found. Setting CardData to empty.


Let's downsample to match our 16 history/social science examples.   

Since we have 4 classes in the original dataset, we'll do 4x8 for our training data. I have cheated got ChatGPT to generate 64 test examples (labelled) for the history/social science examples. So to match this, we'll do 32*4 for the test size!

In [ ]:
# label to stratify 
num_classes = 4
ds = ds.cast_column("label", ClassLabel(num_classes=num_classes))

ds_downsampled = ds.train_test_split(train_size=32,test_size=128, seed=42, stratify_by_column="label")

We'll have to reset the cast_column as we will later combine with our new dataset with two new classes

In [118]:
ds_downsampled = ds_downsampled.cast_column("label", Value("int64"))

Casting the dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/128 [00:00<?, ? examples/s]

Combine:

In [124]:
train_ds = concatenate_datasets([ds_downsampled["train"], new_train_ds])
val_ds = concatenate_datasets([ds_downsampled["test"], new_val_ds])

## Few-Shot Learning with SetFit 
We'll use the package `setfit`, developed by HuggingFace, to do *few-shot* learning which is a a specialized type of ML that relies on limited data! For our model, we'll use the small `all-MiniLM-L6-v2`.

:::{admonition} What is *all-MiniLM-L6-v2* ? 
:class: tip, dropdown
It is a version of *MiniLM* fine-tuned specifically on *sentence embeddings*. `Mini-LM` is `distilled` from 

*Distilliation* is a process of taking a larger language model and compressing it to a smaller size, 
:::


In [170]:
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2", trust_remote_code=True,
)

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Here it becomes a little *bamboozling*, because we are importing `Trainer` from `setfit` which is *close to* but not identicial in parameters to when you write `from transformers import Trainer` ... A bit confusing considering both libraries are from Hugging Face!

In [171]:
args = TrainingArguments(
    batch_size=8,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    metric="accuracy",
    )

In [ ]:
# NB. CELL IS REMOVED FROM BOOK!
import warnings
warnings.filterwarnings("ignore")

In [172]:
trainer.train()

***** Running training *****
  Num unique pairs = 1886
  Batch size = 8
  Num epochs = 1


Step,Training Loss
1,0.543200
50,0.185100
100,0.093500
150,0.042800
200,0.027900


In [167]:
trainer.evaluate()

***** Running evaluation *****


{'accuracy': 0.7637362637362637}